In [ ]:
# ============================================================
# 环境配置
# - Colab 用户：取消注释下方 Colab 区块
# - 本地 Jupyter 用户：直接运行 Local 区块
# ============================================================

# ── Colab 环境（取消注释后运行） ──
# !pip install torch transformers datasets peft accelerate matplotlib numpy -q

# ── 本地 Jupyter 环境 ──
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['torch', 'transformers', 'datasets', 'peft', 'accelerate', 'matplotlib', 'numpy']:
    _install(pkg)

print('Environment ready.')

In [ ]:
import math, random, collections
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, TaskType, get_peft_model

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# LoRA 代码实战：学习实现 vs 工程实现

基于论文 *LoRA: Low-Rank Adaptation of Large Language Models*（Edward J. Hu et al., ICLR 2022），用 **SST-2 二分类情感分析** 任务演示 LoRA 的核心思想。

本 Notebook 包含两条并行路径，使用 **相同的任务和同一份数据集子集**：

| | 学习路径 (Learning) | 工程路径 (Engineering) |
|---|---|---|
| 目标 | 理解 LoRA 到底如何改写线性层 | 掌握 HuggingFace + PEFT 的工业化微调方式 |
| 实现方式 | 手写 `LoRALinear` + 小型文本分类器 | `transformers` + `peft` 对 DistilBERT 做 LoRA 微调 |
| 代码量 | 较多，细节全部显式展开 | 较少，依赖成熟库封装 |
| 适合场景 | 面试准备、原理理解、结构拆解 | 工程落地、快速验证、迁移到真实任务 |

> 两条路径不是两套无关代码，而是同一个 LoRA 思想在不同抽象层级上的表达。

## 一、论文与任务背景

### 1.1 论文信息
- **论文**：*LoRA: Low-Rank Adaptation of Large Language Models*
- **作者**：Edward J. Hu, Yelong Shen, Phillip Wallis, Zeyuan Allen-Zhu, Yuanzhi Li, Shean Wang, Lu Wang, Weizhu Chen
- **会议**：ICLR 2022
- **核心问题**：全参数微调成本太高，是否可以只训练一个低秩更新量 `ΔW`，而冻结原始权重 `W₀`？

### 1.2 为什么用 SST-2 做演示
LoRA 本身不是一个“模型架构”，而是一种**参数高效微调方法**。因此最适合它的教学任务，不是从零训练超大语言模型，而是选一个：
1. 数据可快速下载；
2. CPU / 免费 Colab 都能跑；
3. 又足以展示“冻结大部分参数，只训练少量低秩矩阵”这一核心思想。

SST-2 正好满足这些条件。

### 1.3 架构谱系与本 Notebook 的边界

LoRA 属于 **PEFT（Parameter-Efficient Fine-Tuning）** 家族。它不替代 Transformer、BERT、GPT，而是插在这些模型的线性层上。

本 Notebook 关注：
- LoRA 的公式、参数量、初始化方式；
- 为什么 `ΔW` 可以写成低秩分解；
- 手写版 LoRA 与 HuggingFace / PEFT 的一一对应关系；
- QLoRA 与常见工程扩展。

本 Notebook **不覆盖**：
- 大模型全量 SFT 的完整工程栈；
- 分布式训练；
- 4-bit 量化细节实现（只做概念说明，不在 CPU 教学路径里复现）。

## 二、最小必要理论

LoRA 的核心公式是：

$$
W' = W_0 + \Delta W = W_0 + \frac{\alpha}{r}BA
$$

其中：
- `W₀ ∈ R^{d_out × d_in}`：原始预训练权重，**冻结不动**；
- `A ∈ R^{r × d_in}`、`B ∈ R^{d_out × r}`：新增的两个低秩矩阵，**参与训练**；
- `r`：LoRA rank，通常远小于 `d_in` 和 `d_out`；
- `α / r`：缩放项，用来稳定更新幅度。

如果直接训练 `ΔW`，参数量是 `d_out × d_in`。如果训练 `B` 和 `A`，参数量变成：

$$
\text{params}_{LoRA} = r(d_{in} + d_{out})
$$

当 `r << min(d_in, d_out)` 时，训练参数量会显著下降。

### 初始化为什么常见为 A 随机、B 全零？
因为这样初始时：

$$
\Delta W = BA = 0
$$

也就是说，训练一开始模型行为与原模型完全一致，不会因为插入 LoRA 而立刻破坏已有能力。

### 组件映射表（强制锚点）

| 论文组件 | 学习路径覆盖 | 工程库 / API 对应 | 状态 |
|---|---|---|---|
| 低秩更新 `ΔW = BA` | 手写 `LoRALinear`，显式计算 `B @ A` | `peft.LoraConfig` + `get_peft_model` 在目标层内部注入 | Runnable |
| 缩放项 `α / r` | 在 `forward()` 中显式乘上 `self.scaling` | `LoraConfig(lora_alpha=..., r=...)` | Runnable |
| 原始权重冻结 `W₀` | `requires_grad=False` | PEFT 自动冻结 backbone 主体参数 | Runnable |
| A 随机、B 零初始化 | 手写初始化逻辑 | PEFT 内部初始化策略 | Runnable |
| 只改线性层，不改整体架构 | 只替换一个线性映射 | `target_modules=['q_lin', 'v_lin']` | Runnable |
| 参数高效 vs 全量微调 | 显式计算参数量对比 | `model.print_trainable_parameters()` | Runnable |
| LoRA 常挂在 attention / FFN 线性层 | 学习路径用单层线性映射做最小演示 | DistilBERT attention 中的 `q_lin` / `v_lin` | Runnable |
| QLoRA（量化 + LoRA） | 概念说明，不在 CPU 路径复现 | bitsandbytes + PEFT 组合 | Explain-only |

## 1. 数据准备

这里使用 HuggingFace `datasets` 直接加载 `glue/sst2`。为了保证免费 Colab 和 CPU 都能稳定运行，我们只取一个小子集：
- 训练集：2000 条
- 验证集：500 条

这样既足够看到 LoRA 的作用，也不会让工程路径训练时间过长。

In [ ]:
raw = load_dataset('glue', 'sst2')
train_raw = raw['train'].shuffle(seed=42).select(range(2000))
eval_raw = raw['validation'].shuffle(seed=42).select(range(500))

print(train_raw)
print(eval_raw)
print(train_raw[0])

In [ ]:
MAX_VOCAB = 4000

def simple_tokenize(text):
    return [tok.lower() for tok in text.split()]

counter = collections.Counter()
for ex in train_raw:
    counter.update(simple_tokenize(ex['sentence']))

vocab = {'<unk>': 0}
for word, _ in counter.most_common(MAX_VOCAB - 1):
    vocab[word] = len(vocab)

VOCAB_SIZE = len(vocab)

def vectorize(text):
    vec = torch.zeros(VOCAB_SIZE, dtype=torch.float32)
    tokens = simple_tokenize(text)
    for tok in tokens:
        vec[vocab.get(tok, 0)] += 1.0
    if tokens:
        vec /= len(tokens)
    return vec

train_x = torch.stack([vectorize(ex['sentence']) for ex in train_raw])
train_y = torch.tensor(train_raw['label'], dtype=torch.long)
eval_x = torch.stack([vectorize(ex['sentence']) for ex in eval_raw])
eval_y = torch.tensor(eval_raw['label'], dtype=torch.long)

train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=64, shuffle=True)
eval_loader = DataLoader(TensorDataset(eval_x, eval_y), batch_size=128, shuffle=False)

print('VOCAB_SIZE =', VOCAB_SIZE)

In [ ]:
sample_x, sample_y = next(iter(train_loader))
print('Feature batch shape:', sample_x.shape)
print('Label batch shape:', sample_y.shape)
print('Example sentence:', train_raw[0]['sentence'])
print('Example label   :', train_raw[0]['label'])

In [ ]:
# ── 超参数（两条路径共用，集中管理） ──
HIDDEN_DIM = 128
NUM_CLASSES = 2
LORA_R = 8
LORA_ALPHA = 16
DROPOUT = 0.1
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 10

HF_MODEL_NAME = 'distilbert-base-uncased'
HF_MAX_LENGTH = 128
HF_BATCH_SIZE = 16
HF_EVAL_BATCH_SIZE = 32
HF_NUM_EPOCHS = 1
HF_LR = 2e-4

In [ ]:
def accuracy_from_logits(logits, labels):
    preds = logits.argmax(dim=-1)
    return (preds == labels).float().mean().item()

def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits.detach(), y)
    return total_loss / len(loader), total_acc / len(loader)

@torch.no_grad()
def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, y)
    return total_loss / len(loader), total_acc / len(loader)

def plot_learning_curves(history, title='Learning path curves'):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], marker='o', label='train_loss')
    plt.plot(epochs, history['eval_loss'], marker='s', label='eval_loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Loss curves')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_acc'], marker='o', label='train_acc')
    plt.plot(epochs, history['eval_acc'], marker='s', label='eval_acc')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Accuracy curves')
    plt.legend()
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

## 2. 学习路径：手写 LoRA

### 2.1 组件一：LoRA 线性层

对一个普通线性层而言，原本输出是：

$$
y = xW_0^T
$$

加入 LoRA 后变成：

$$
y = x(W_0 + \frac{\alpha}{r}BA)^T
$$

这里的输入 / 输出 shape 为：
- 输入：`x` → `(batch, d_in)`
- 原始权重：`W₀` → `(d_out, d_in)`
- `A` → `(r, d_in)`
- `B` → `(d_out, r)`
- 输出：`y` → `(batch, d_out)`

为什么这招有效？因为很多下游任务真正需要的，不是“把整块权重矩阵彻底改写”，而是在原有表示空间上做一个**低维方向的修正**。

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, r=8, alpha=16, bias=False):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r
        self.weight = nn.Parameter(torch.empty(out_features, in_features), requires_grad=False)
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        self.bias = None
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features), requires_grad=False)
        self.lora_A = nn.Parameter(torch.empty(r, in_features))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))

    def forward(self, x):
        delta_w = (self.lora_B @ self.lora_A) * self.scaling
        weight_eff = self.weight + delta_w
        return torch.nn.functional.linear(x, weight_eff, self.bias)

### 2.2 组件二：最小可训练文本分类器

为了让 CPU 也能跑通，我们不用完整 Transformer，而是构造一个**最小教学模型**：
1. 先把一句话转成 bag-of-words 向量；
2. 通过一个 **冻结的线性层 + LoRA 低秩更新** 映射到隐藏空间；
3. 经过 `LayerNorm + GELU + Dropout`；
4. 再接一个分类头得到二分类 logits。

这不是 LoRA 的典型工业用法，但它忠实保留了 LoRA 的核心机制。

In [ ]:
class TinyLoRAClassifier(nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_classes, r, alpha, dropout=0.1):
        super().__init__()
        self.proj = LoRALinear(vocab_size, hidden_dim, r=r, alpha=alpha, bias=False)
        self.norm = nn.LayerNorm(hidden_dim)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        hidden = self.proj(x)
        hidden = self.norm(hidden)
        hidden = self.act(hidden)
        hidden = self.dropout(hidden)
        logits = self.classifier(hidden)
        return logits

learning_model = TinyLoRAClassifier(
    vocab_size=VOCAB_SIZE,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    r=LORA_R,
    alpha=LORA_ALPHA,
    dropout=DROPOUT,
).to(device)

trainable, total = count_trainable_params(learning_model)
full_linear_params = VOCAB_SIZE * HIDDEN_DIM
lora_linear_params = LORA_R * (VOCAB_SIZE + HIDDEN_DIM)

print(learning_model)
print(f'Trainable params: {trainable:,}')
print(f'Total params    : {total:,}')
print(f'Full linear params without LoRA: {full_linear_params:,}')
print(f'LoRA params for that layer     : {lora_linear_params:,}')

### 2.3 为什么 `α / r` 很重要？

如果没有缩放项，`r` 变大时，`BA` 的数值尺度也可能明显变化。LoRA 通过 `α / r` 把更新幅度稳定在可控范围内，让 rank 的变化更像是在调“表达容量”，而不是直接把数值尺度搞乱。

同时，`A` 随机、`B` 全零的初始化也保证了训练起点满足 `W' = W_0`。

In [ ]:
with torch.no_grad():
    delta = learning_model.proj.lora_B @ learning_model.proj.lora_A
    print('Effective delta shape:', delta.shape)
    print('Theoretical rank upper bound:', LORA_R)
    print('Initial delta Frobenius norm:', delta.norm().item())

### 2.4 训练学习路径模型

下面开始真正训练。这里训练的参数包括：
- LoRA 的 `A` / `B`；
- 最后的分类头。

冻结的基础权重 `W₀` 不参与更新。

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    [p for p in learning_model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

learning_history = {
    'train_loss': [],
    'eval_loss': [],
    'train_acc': [],
    'eval_acc': [],
}

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_epoch(learning_model, train_loader, optimizer, criterion)
    eval_loss, eval_acc = evaluate_model(learning_model, eval_loader, criterion)
    learning_history['train_loss'].append(train_loss)
    learning_history['eval_loss'].append(eval_loss)
    learning_history['train_acc'].append(train_acc)
    learning_history['eval_acc'].append(eval_acc)
    print(
        f'Epoch {epoch:02d} | ' +
        f'train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | ' +
        f'eval_loss={eval_loss:.4f} | eval_acc={eval_acc:.4f}'
    )

plot_learning_curves(learning_history, title='Learning path: Tiny LoRA classifier')

In [ ]:
@torch.no_grad()
def predict_learning(texts):
    learning_model.eval()
    feats = torch.stack([vectorize(t) for t in texts]).to(device)
    logits = learning_model(feats)
    probs = torch.softmax(logits, dim=-1).cpu()
    preds = probs.argmax(dim=-1).tolist()
    return preds, probs

label_names = {0: 'negative', 1: 'positive'}
sample_texts = [eval_raw[i]['sentence'] for i in range(8)]
sample_labels = [eval_raw[i]['label'] for i in range(8)]
learning_preds, learning_probs = predict_learning(sample_texts)

for text, gold, pred, prob in zip(sample_texts, sample_labels, learning_preds, learning_probs):
    print('-' * 80)
    print(text)
    print(f'gold={label_names[gold]} | pred={label_names[pred]} | prob_positive={prob[1].item():.4f}')

## 3. 工程路径：HuggingFace + PEFT

### 3.1 为什么这里用 DistilBERT + PEFT

LoRA 在工业场景中通常不是搭配“随机初始化的小模型”，而是搭配**已经预训练好的 backbone**。这里选 `distilbert-base-uncased` 的原因是：
- 体量比 `bert-base-uncased` 更轻；
- 文本分类场景成熟；
- 免费 Colab 更容易在可接受时间内跑完。

这一路径对应的决策是：
- **学习路径深度**：L1（完整可训练小模型）
- **工程路径形态**：E1（成熟库）
- **工程动作**：Partial fine-tune（只训练 LoRA + 分类头）

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch['sentence'], truncation=True, max_length=HF_MAX_LENGTH)

hf_train = train_raw.map(tokenize_batch, batched=True)
hf_eval = eval_raw.map(tokenize_batch, batched=True)
remove_cols = [c for c in hf_train.column_names if c not in ['input_ids', 'attention_mask', 'label']]
hf_train = hf_train.remove_columns(remove_cols)
hf_eval = hf_eval.remove_columns(remove_cols)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
hf_train.set_format('torch')
hf_eval.set_format('torch')
print(hf_train[0].keys())
print('Tokenized train size:', len(hf_train))
print('Tokenized eval size :', len(hf_eval))

### 3.2 微调策略

这里的工程思路不是“全参数微调”，而是：
1. 加载一个已经预训练好的 DistilBERT；
2. 把 LoRA 插到 self-attention 的线性投影层；
3. 只训练 LoRA 参数和分类头；
4. 其余 backbone 主体保持冻结。

在 `TaskType.SEQ_CLS` 下，PEFT 会按序列分类任务的语义去包装模型。

In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    HF_MODEL_NAME,
    num_labels=NUM_CLASSES,
)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=DROPOUT,
    target_modules=['q_lin', 'v_lin'],
    bias='none',
)

peft_model = get_peft_model(base_model, peft_config)
peft_model.to(device)
peft_model.print_trainable_parameters()

### 3.3 学习路径与工程路径的内部对应

| 学习路径实现 | 工程库内部对应 | 说明 |
|---|---|---|
| `LoRALinear` 里显式维护 `W₀ + (α/r)BA` | PEFT 注入到 `q_lin` / `v_lin` | 同一机制，更高抽象 |
| `requires_grad=False` 冻结原权重 | PEFT 默认冻结 backbone 主体 | 同一训练策略 |
| 手动统计参数量 | `print_trainable_parameters()` | 同一指标，不同接口 |
| 自写训练循环 | `Trainer` | 同一优化过程，被库封装 |
| 手写分类输出 | `AutoModelForSequenceClassification` 的分类头 | 同一任务头，不同复用层级 |

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': float((preds == labels).mean())}

training_args = TrainingArguments(
    output_dir='./tmp_lora_distilbert',
    eval_strategy='epoch',
    save_strategy='no',
    logging_strategy='steps',
    logging_steps=20,
    learning_rate=HF_LR,
    per_device_train_batch_size=HF_BATCH_SIZE,
    per_device_eval_batch_size=HF_EVAL_BATCH_SIZE,
    num_train_epochs=HF_NUM_EPOCHS,
    weight_decay=0.01,
    report_to=[],
    remove_unused_columns=False,
    seed=42,
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_eval,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
hf_metrics = trainer.evaluate()
print(hf_metrics)

In [ ]:
pred_block = trainer.predict(hf_eval.select(range(8)))
hf_preds = np.argmax(pred_block.predictions, axis=-1).tolist()

for i, pred in enumerate(hf_preds):
    text = eval_raw[i]['sentence']
    gold = eval_raw[i]['label']
    print('-' * 80)
    print(text)
    print(f'gold={label_names[gold]} | pred={label_names[pred]}')

hf_log_history = trainer.state.log_history
hf_train_loss = [x['loss'] for x in hf_log_history if 'loss' in x]
hf_eval_acc = [x['eval_accuracy'] for x in hf_log_history if 'eval_accuracy' in x]
print('Logged training loss points:', len(hf_train_loss))
print('Logged eval accuracy points:', len(hf_eval_acc))

## 4. 学习路径 vs 工程路径对比

| 对比维度 | 学习路径 | 工程路径 |
|---|---|---|
| 教育价值 | 能看到 `W₀`、`A`、`B`、`α/r` 如何真正参与计算 | 能理解工业级 API 如何把 LoRA 落到真实预训练模型上 |
| 代码量 | 较多，细节完全展开 | 较少，核心接口高度封装 |
| 灵活性 | 可以随时改公式、初始化、结构 | 受限于库暴露的接口与目标模块命名 |
| 稳定性 | 容易因 shape / 初始化 / 学习率而不稳 | 成熟库验证更多，默认更稳 |
| 工业适配度 | 适合研究原型、教学推导、面试讲解 | 适合业务微调、实验复现、快速迁移 |
| 适用场景 | 想深入理解 LoRA 原理；想手推公式到代码；想做结构级实验 | 想把 LoRA 接到现成模型；想快速跑任务；想迁移到真实 NLP pipeline |

In [ ]:
compare_texts = [eval_raw[i]['sentence'] for i in range(5)]
learning_preds, _ = predict_learning(compare_texts)

peft_model.eval()
with torch.no_grad():
    hf_inputs = tokenizer(compare_texts, return_tensors='pt', padding=True, truncation=True, max_length=HF_MAX_LENGTH)
    hf_inputs = {k: v.to(device) for k, v in hf_inputs.items()}
    hf_logits = peft_model(**hf_inputs).logits
    hf_preds_direct = hf_logits.argmax(dim=-1).cpu().tolist()

for i, text in enumerate(compare_texts):
    print('-' * 100)
    print(text)
    print(
        f'gold={label_names[eval_raw[i]["label"]]} | ' +
        f'learning={label_names[learning_preds[i]]} | ' +
        f'engineering={label_names[hf_preds_direct[i]]}'
    )

plt.figure(figsize=(8, 4))
plt.plot(learning_history['train_loss'], marker='o', label='learning_train_loss')
plt.plot(learning_history['eval_loss'], marker='s', label='learning_eval_loss')
if hf_train_loss:
    plt.plot(np.linspace(0, max(1, len(learning_history['train_loss']) - 1), len(hf_train_loss)), hf_train_loss, marker='^', label='engineering_train_loss')
plt.xlabel('Epoch / logged step index')
plt.ylabel('Loss')
plt.title('Learning vs engineering loss traces')
plt.legend()
plt.show()

## 5. 训练阶段 vs 推理阶段

虽然情感分类不像自回归生成那样存在 teacher forcing / greedy decoding 的强差异，但 LoRA 依然区分了训练与推理：

| 阶段 | 学习路径行为 | 工程路径行为 |
|---|---|---|
| 训练 | 更新 `A` / `B` 与分类头；`W₀` 冻结 | `Trainer.train()` 更新 LoRA adapter 与分类头；backbone 主体冻结 |
| 推理 | 仅前向计算 `W₀ + (α/r)BA`，不再反向传播 | `model.eval()` 后只做前向；LoRA 已成为有效权重修正 |
| Dropout | 训练时开启 | 推理时关闭 |

关键洞见：**两条路径执行的是同一个算法，只是抽象层级不同。**

## 6. 结果与边界

### 学习路径的收获
- 你可以非常明确地知道 LoRA 改的不是“整个模型”，而是某些线性层上的低秩增量；
- 你可以直接解释 `r`、`α`、初始化策略、参数量缩减公式；
- 你也能看清楚：LoRA 不是魔法，它依赖于一个已经足够好的底座 `W₀`。

### 工程路径的收获
- 你可以几行代码把 LoRA 接到真实预训练模型上；
- 可以直接迁移到更多文本分类、生成、指令微调任务；
- 也更容易升级到 QLoRA、8-bit / 4-bit 推理、批处理推理等真实工程设置。

### 边界
- 学习路径的模型不是预训练 Transformer，因此它演示的是 **LoRA 机制本身**，不是大模型能力上限；
- 工程路径虽然更真实，但很多细节都藏进了库内部，不适合第一次学习时直接建立心智模型；
- QLoRA 在真实大模型上非常重要，但它依赖量化库与更复杂的硬件 / 内存环境，不适合这个 CPU 友好教学 notebook 直接复刻。

In [ ]:
with torch.no_grad():
    delta = (learning_model.proj.lora_B @ learning_model.proj.lora_A).detach().cpu()
    singular_values = torch.linalg.svdvals(delta)

plt.figure(figsize=(6, 4))
plt.plot(singular_values.numpy(), marker='o')
plt.xlabel('Index')
plt.ylabel('Singular value')
plt.title('Singular values of learned LoRA delta')
plt.show()

## 7. 面试与项目表达

### 高频面试题

**Q1：LoRA 为什么能比全参数微调更省资源？**
- 因为它不直接更新完整的 `ΔW`，而是把更新写成低秩分解 `BA`；
- 训练参数量从 `d_out × d_in` 变成 `r(d_in + d_out)`；
- 同时 backbone 主体冻结，梯度与优化器状态只需要为少量参数保存。

**Q2：为什么常见初始化是 A 随机、B 为 0？**
- 因为这样一开始 `BA = 0`；
- 初始模型行为与原模型一致，更稳定；
- 训练过程中再逐步学习偏移，而不是在第 0 步就破坏预训练能力。

**Q3：`r` 和 `alpha` 分别在控制什么？**
- `r` 主要控制低秩更新的容量上限；
- `alpha` 控制更新尺度；
- 更高的 `r` 不一定更好，过高会增加参数量与优化难度。

**Q4：LoRA 一般挂在哪些层？为什么？**
- 最常见是 attention 的 `q / k / v / o` 投影层，以及部分 FFN 线性层；
- 因为这些地方正是表示变换最集中的线性映射；
- 改这里通常比改 embedding 或全部参数更高效。

**Q5：LoRA 和 Adapter / Prefix Tuning 的主要区别是什么？**
- LoRA 是对权重增量做低秩分解；
- Adapter 是在网络中插入额外瓶颈模块；
- Prefix / Prompt Tuning 更偏向输入侧或注意力上下文侧的可训练提示。

**Q6：QLoRA 比 LoRA 多做了什么？**
- QLoRA 先把底座模型量化（典型是 4-bit），再在其上训练 LoRA adapter；
- 关键目标是进一步降低显存占用；
- 它不改变 LoRA 的数学核心，但改变了底座权重的存储与计算方式。

**Q7：LoRA 会不会让训练一定更快？**
- 不一定。它通常**更省显存和可训练参数**，但 wall-clock time 不总是线性下降；
- 在小模型或特定实现里，额外 adapter 路径本身也会引入开销；
- 所以更准确的说法是：LoRA 的核心优势首先是 **参数效率与内存效率**。

### 面试速答提纲

| # | 问题 | 一句话回答 |
|---|---|---|
| 1 | LoRA 本质是什么？ | 在冻结原权重的前提下，用低秩矩阵近似权重更新量。 |
| 2 | 为什么参数更少？ | 因为训练的是 `B` 和 `A`，不是完整 `ΔW`。 |
| 3 | `alpha / r` 的作用？ | 让 rank 变化时更新尺度更稳定。 |
| 4 | 为什么 A 随机、B 为 0？ | 保证初始时 `ΔW = 0`，不破坏原模型。 |
| 5 | QLoRA 是什么？ | 量化后的底座模型 + LoRA adapter 微调。 |
| 6 | LoRA 常挂哪里？ | attention 和 FFN 的线性投影层。 |
| 7 | LoRA vs 全参微调？ | 性能接近时更省内存、更省存储，但不是总更快。 |

### 项目表达

> 如果面试官问“你做过什么 LoRA 相关项目”，可以这样组织回答：

- **学习深度**：我从零实现了 `LoRALinear`，明确验证了 `W' = W₀ + (α/r)BA` 的代码路径与参数量缩减公式。
- **工程能力**：我用 HuggingFace `transformers + peft` 在 SST-2 上对 DistilBERT 做了 LoRA 微调，能解释 `target_modules`、`TaskType.SEQ_CLS`、`Trainer` 的作用。
- **对比思考**：我能区分教学路径与工程路径：前者适合理解内部机制，后者适合迁移到真实业务任务。

### 延伸阅读与对比

| 对比维度 | Full Fine-Tuning | LoRA | QLoRA |
|---|---|---|---|
| 核心思想 | 更新全部参数 | 更新低秩 adapter | 量化底座后更新低秩 adapter |
| 显存压力 | 高 | 低很多 | 更低 |
| 实现复杂度 | 中 | 中 | 更高 |
| 适用场景 | 资源充足、极致定制 | 常规 PEFT 微调 | 大模型受限显存微调 |

### 进阶探索方向
- **方向 1：把 LoRA 挂到更多模块** —— 对比只改 attention 与 attention+FFN 的差异。
- **方向 2：系统扫 rank / alpha** —— 研究容量、稳定性、效果之间的 trade-off。
- **方向 3：升级到 QLoRA** —— 在更大模型上比较 LoRA 与量化 LoRA 的显存收益。
- **方向 4：合并 adapter** —— 研究训练后是否将 LoRA 权重 merge 回原模型。

### 参考来源
- Hu et al., 2022, *LoRA: Low-Rank Adaptation of Large Language Models*, OpenReview / ICLR 2022.
- Dettmers et al., 2023, *QLoRA: Efficient Finetuning of Quantized LLMs*, arXiv:2305.14314.
- Hugging Face Transformers documentation, current Trainer / AutoModel / AutoTokenizer docs.
- Hugging Face PEFT documentation, current `LoraConfig` / `get_peft_model` docs。